# Automotive Regulation and Type Approval for Automated Driving

**Module 08 · Standards, Regulation and Safety Case** · Notebook 2 of 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/08_Standards_Regulation_and_Safety_Case/02_regulation_and_type_approval.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~60 min | [Standards integration](01_standards_integration.ipynb), [ISO/SAE 21434 fundamentals](../05_Cybersecurity_ISO21434/01_iso21434_fundamentals.ipynb) | UN R155, R156, R157, UN ADS Regulation (2026); Reg. (EU) 2018/858, 2019/2144, 2022/1426, 2024/1689; ISO/SAE 21434:2021, ISO 24089:2023 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Distinguish type approval (UNECE / EU / China) from self-certification (US) and name the actors involved
- Place the main AV regulations on a dated global timeline and say what each requires
- Map UN R155 CSMS processes to ISO/SAE 21434 clauses and UN R156 SUMS to ISO 24089
- State the UN R157 (ALKS) transition-demand timings and model them
- Explain correctly how the EU AI Act reaches vehicles (Art. 2(2), Annex I Section B), including the 2026 change of dates
- Assemble an evidence checklist for an SAE Level 3 motorway system seeking EU approval

> **Time-sensitive content.** Regulation changes quickly. Every dated statement below carries "(as of October 2026)" or a date, plus a source. Check the source before relying on it for a real programme. This notebook is teaching material, not legal advice.

In [ ]:
from dataclasses import dataclass, field
from datetime import date
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

OI = {"orange": "#E69F00", "sky": "#56B4E9", "green": "#009E73", "yellow": "#F0E442",
      "blue": "#0072B2", "vermillion": "#D55E00", "purple": "#CC79A7", "black": "#000000", "grey": "#999999"}
pd.set_option("display.max_colwidth", 100)
AS_OF = "October 2026"
print(f"Regulatory analysis tools loaded (content as of {AS_OF})")

## 1. Type approval versus self-certification

| | **Type approval** (UNECE 1958 Agreement, EU, China, Japan, …) | **Self-certification** (United States) |
|---|---|---|
| Who decides before sale? | A **type-approval authority** (e.g. KBA in Germany, VCA in Great Britain) grants approval after tests and audits by a designated **technical service** (e.g. TÜV, DEKRA, UTAC) | Nobody: the manufacturer certifies compliance with the Federal Motor Vehicle Safety Standards (FMVSS) |
| Legal basis | UN Regulations under the 1958 Agreement (mutual recognition among Contracting Parties applying them); EU framework Reg. (EU) 2018/858 and General Safety Regulation (EU) 2019/2144 | 49 U.S.C. Chapter 301; FMVSS in 49 CFR Part 571 |
| After sale | Conformity of production; market surveillance; changes may need an extension of the approval | NHTSA enforcement, defect investigations, recalls; exemptions (49 CFR Part 555) for non-compliant designs |
| Software / AI | Management-system audits (CSMS, SUMS, safety management for ALKS/ADS) plus vehicle-type assessment | No pre-market review; reporting obligations such as the Standing General Order |

UN Regulations under the **1958 Agreement** support reciprocal recognition of approvals. **UN GTRs** under the **1998 Agreement** (to which the US, China and others are party) harmonise technical requirements but carry no mutual recognition of approvals.

China combines mandatory national standards (GB) with MIIT product access ("product announcement") and CCC certification.

---

## 2. Global regulatory timeline (as of October 2026)

The timeline is built from a table of dated events, each with a source link. It is not exhaustive. Where a date comes from secondary reporting rather than the official text, the `basis` column says so.

In [ ]:
events = pd.DataFrame([
    ("2021-01-22", "UNECE", "UN R155, R156 and R157 (ALKS, 60 km/h) enter into force", "official",
     "https://unece.org/sites/default/files/2025-06/R157r1e.pdf"),
    ("2021-07-28", "Germany", "Act on Autonomous Driving (StVG §§1d-1l): Level 4 in defined operating areas", "official",
     "https://www.gesetze-im-internet.de/stvg/__1d.html"),
    ("2022-06-24", "Germany", "AFGBV ordinance: approval and operation of autonomous vehicles", "official",
     "https://www.gesetze-im-internet.de/afgbv/"),
    ("2022-06-22", "UNECE", "WP.29 adopts R157 01 series: up to 130 km/h, lane change (in force Jan 2023)", "secondary",
     "https://www.tuvsud.com/en-gb/industries/mobility-and-automotive/automotive-and-oem/autonomous-driving/compliance-with-new-automated-lane-keeping-system-regulation"),
    ("2022-07-06", "EU", "R155/R156 mandatory for new vehicle types (Reg. (EU) 2019/2144)", "official",
     "https://eur-lex.europa.eu/eli/reg/2019/2144/oj"),
    ("2022-08-05", "EU", "Implementing Reg. (EU) 2022/1426: ADS of fully automated vehicles, small series", "official",
     "https://eur-lex.europa.eu/eli/reg_impl/2022/1426/oj"),
    ("2024-05-20", "UK", "Automated Vehicles Act 2024 receives Royal Assent", "official",
     "https://www.legislation.gov.uk/ukpga/2024/10/contents"),
    ("2024-07-07", "EU", "R155/R156 mandatory for all new vehicles registered", "official",
     "https://eur-lex.europa.eu/eli/reg/2019/2144/oj"),
    ("2024-08-01", "EU", "AI Act (Reg. (EU) 2024/1689) enters into force", "official",
     "https://eur-lex.europa.eu/eli/reg/2024/1689/oj"),
    ("2025-04-24", "US", "DOT/NHTSA AV Framework; Third Amended SGO (effective 16 Jun 2025)", "secondary",
     "https://www.mayerbrown.com/en/insights/publications/2025/04/dot-and-nhtsa-announce-autonomous-vehicle-framework"),
    ("2025-12-15", "China", "MIIT grants first conditional L3 product access (Changan Deepal SL03, BAIC Arcfox Alpha S)", "secondary",
     "https://cnevpost.com/2025/12/15/china-grants-1st-l3-autonomous-driving-permits-passenger-cars/"),
    ("2026-01-01", "China", "Mandatory GB 44495 (cybersecurity), GB 44496 (software update), GB 44497 (DSSAD) apply", "secondary",
     "https://www.tuvsud.com/en/knowledge-hub/technical-updates/consumer-products-and-retail-essentials/china-implements-vehicle-cybersecurity-technical-requirements"),
    ("2026-03-03", "EU", "Implementing Reg. (EU) 2026/481 amends 2022/1426", "official",
     "https://eur-lex.europa.eu/eli/reg_impl/2026/481/oj"),
    ("2026-05-22", "UK", "Automated passenger services permitting scheme opens (pilots)", "secondary",
     "https://www.gov.uk/government/speeches/automated-passenger-services-permitting-scheme"),
    ("2026-06-01", "UK", "GB type approval: cyber security and software update rules for new types", "official",
     "https://www.vehicle-certification-agency.gov.uk/blog/new-gb-type-approval-legislation-cyber-security-software-updates-introduced"),
    ("2026-06-24", "UNECE", "WP.29 adopts UN Regulation and UN GTR on Automated Driving Systems", "official",
     "https://unece.org/sustainable-development/press/unece-adopts-first-ever-global-rules-allowing-fully-autonomous"),
    ("2026-06-26", "US", "NHTSA proposes FMVSS 135 changes for ADS vehicles; AV STEP proposal withdrawn", "secondary",
     "https://www.crowell.com/en/insights/client-alerts/nhtsa-proposes-updates-to-federal-brake-standards-for-autonomous-vehicles-and-withdraws-av-step-program"),
    ("2026-07-27", "EU", "Reg. (EU) 2026/1744 (Digital Omnibus on AI) enters into force: high-risk dates moved", "official",
     "https://eur-lex.europa.eu/eli/reg/2026/1744/oj/eng"),
    ("2026-12-09", "EU", "Product Liability Directive (EU) 2024/2853 applies (software is a product)", "official",
     "https://eur-lex.europa.eu/eli/dir/2024/2853/oj"),
    ("2027-12-02", "EU", "AI Act: Annex III high-risk obligations apply (per 2026/1744)", "official",
     "https://eur-lex.europa.eu/eli/reg/2026/1744/oj/eng"),
    ("2028-08-02", "EU", "AI Act: Art. 6(1)/Annex I high-risk obligations apply (per 2026/1744)", "official",
     "https://eur-lex.europa.eu/eli/reg/2026/1744/oj/eng"),
], columns=["date", "jurisdiction", "event", "basis", "source"])
events["date"] = pd.to_datetime(events["date"])
events = events.sort_values("date").reset_index(drop=True)

# Short labels for the figure (full text in the table and the printed summary)
SHORT = dict(zip(events["event"], [
    "R155/R156/R157 in force", "StVG L4 law", "R157 01 series", "AFGBV", "R155/R156 new types",
    "EU 2022/1426", "AV Act 2024", "R155/R156 all new", "AI Act in force", "AV Framework / SGO",
    "First L3 approvals", "GB 44495/96/97", "EU 2026/481", "APS permits", "GB type approval cyber",
    "UN ADS Regulation", "FMVSS 135 NPRM", "AI Omnibus 2026/1744", "PLD applies",
    "AI Act Annex III", "AI Act Annex I"]))

lanes = ["UNECE", "EU", "Germany", "UK", "US", "China"]
lane_colour = dict(zip(lanes, [OI["blue"], OI["orange"], OI["grey"], OI["purple"], OI["vermillion"], OI["green"]]))
today = pd.Timestamp("2026-10-28")

fig, ax = plt.subplots(figsize=(15, 7))
for lane_idx, lane in enumerate(lanes):
    sub = events[events["jurisdiction"] == lane]
    y = len(lanes) - 1 - lane_idx
    ax.axhline(y, color="lightgrey", lw=0.8, zorder=0)
    for k, (_, r) in enumerate(sub.iterrows()):
        marker = "o" if r["date"] <= today else "D"
        ax.scatter(r["date"], y, s=60, color=lane_colour[lane], marker=marker, edgecolor="black", zorder=3)
        ax.annotate(SHORT.get(r["event"], r["event"][:30]), (r["date"], y), xytext=(4, [9, -15, 21][k % 3]),
                    textcoords="offset points", fontsize=7.5)
ax.axvline(today, color="black", ls="--", lw=1)
ax.text(today, len(lanes) - 0.4, " as of Oct 2026", fontsize=8)
ax.set_yticks(range(len(lanes)))
ax.set_yticklabels(lanes[::-1])
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlim(pd.Timestamp("2020-10-01"), pd.Timestamp("2029-07-01"))
ax.set_ylim(-0.7, len(lanes) - 0.2)
ax.set_xlabel("Date")
ax.set_title("Automated-driving regulation milestones (circles: past, diamonds: scheduled), as of October 2026")
plt.tight_layout()
plt.show()

# Text summary for screen readers (derived from the table)
for _, r in events.iterrows():
    tag = "" if r["basis"] == "official" else " [secondary source]"
    print(f"{r['date']:%Y-%m-%d}  {r['jurisdiction']:<8} {r['event']}{tag}")

**Notes on the timeline (as of October 2026):**

- **UN R155/R156 in the EU.** Reg. (EU) 2019/2144 made both regulations mandatory for new vehicle types from 6 July 2022 and for all new vehicles from 7 July 2024. Other 1958 Agreement Contracting Parties that apply them include Japan. Great Britain set 1 June 2026 as the first mandatory date for new types under GB type approval.
- **UN R157 (ALKS).** The original 2021 text limited the system to 60 km/h on motorway-type roads. The **01 series** (adopted June 2022, applying from January 2023) raised the limit to **130 km/h** and added **system-initiated lane changes**.
- **UN ADS Regulation (24 June 2026).** WP.29 adopted a UN Regulation (1958 Agreement) and a parallel UN GTR (1998 Agreement) on Automated Driving Systems. According to UNECE, the Regulation requires:
  - an audited **safety management system**;
  - **credible testing**, including credibility criteria for virtual toolchains;
  - a **safety case** showing no unreasonable risk, with performance at least that of a competent, careful human driver;
  - **in-service monitoring and reporting**;
  - **data recording**.

  About 90 existing UN Regulations were amended so that they apply to vehicles without conventional driver controls. UNECE said it would enter into force about one month after adoption. Check the UNECE status list for the formal date and the Regulation number ([UNECE press release](https://unece.org/sustainable-development/press/unece-adopts-first-ever-global-rules-allowing-fully-autonomous); [TÜV Rheinland, Jul 2026](https://www.tuv.com/content-media-files/master-content/press/pdf-word-en/2026_07/tuev-rheinland-new-un-regulation-and-global-technical-regulation-for-automated-driving-systems-en.pdf)). In the EU, Level 4 manufacturers are expected to be able to choose between EU 2022/1426 and the UN route (TÜV Rheinland).
- **EU 2022/1426.** This regulation covers type approval of the ADS of **fully automated vehicles** for specific use cases (e.g. hub-to-hub, shuttles, automated valet parking), produced in **small series**. Implementing Reg. (EU) 2026/481 amended it.
- **Germany.** The 2021 Act on Autonomous Driving (StVG §§1d–1l) and the **AFGBV** (24 June 2022) allow Level 4 operation in approved operating areas under a **technical supervisor** (*Technische Aufsicht*). The KBA issues the vehicle permits and the federal states approve the operating areas.
- **UK.** The Automated Vehicles Act 2024 (Royal Assent 20 May 2024) creates authorisation, operator licensing and in-use regulation. The Government targets full implementation in the second half of 2027. Automated passenger service (APS) pilots were brought forward to 2026 ([gov.uk written statement](https://www.gov.uk/government/speeches/automated-vehicles-act-2024-implementation); [APS permitting scheme](https://www.gov.uk/government/speeches/automated-passenger-services-permitting-scheme)). Secondary sources report that applications opened in May 2026. Whether paid driverless services have started is not confirmed here.
- **US.** There is no federal pre-market approval of ADS.
  - **Standing General Order 2021-01** requires crash reporting for ADS and Level 2 ADAS. The Third Amended SGO has been effective since 16 June 2025, with initial reports for the most serious crashes due within 5 calendar days. NHTSA sought reinstatement of the information collection in May 2026 ([NHTSA SGO page](https://www.nhtsa.gov/laws-regulations/standing-general-order-crash-reporting)).
  - The April 2025 **AV Framework** extended the exemption programme for research and demonstration vehicles (non-commercial) to domestically built vehicles and streamlined Part 555 exemptions.
  - In 2026 NHTSA proposed FMVSS amendments for ADS vehicles without manual controls (e.g. FMVSS 102, 103/104 and 135) and **withdrew the AV STEP proposal**. All of these were proposals, not final rules, as of October 2026 ([Federal Register, FMVSS 102 NPRM](https://www.federalregister.gov/documents/2026/03/16/2026-05024/federal-motor-vehicle-safety-standards-modernization-of-fmvss-no-102-to-accommodate-ads-equipped); [Crowell, Jul 2026](https://www.crowell.com/en/insights/client-alerts/nhtsa-proposes-updates-to-federal-brake-standards-for-autonomous-vehicles-and-withdraws-av-step-program)).
- **China.**
  - Mandatory **GB 44495-2024** (vehicle cybersecurity), **GB 44496-2024** (software update) and **GB 44497-2024** (data storage for automated driving) apply from 1 January 2026. They mirror R155 and R156; the transition for already-approved types is not verified here.
  - The first **conditional L3** product approvals came on 15 December 2025: Deepal SL03 (single-lane, ≤ 50 km/h in congestion on designated Chongqing roads) and Arcfox Alpha S (≤ 80 km/h on designated Beijing roads).
  - MIIT said in October 2026 that a second batch was in preparation ([CnEVPost, 22 Oct 2026](https://cnevpost.com/2026/10/22/china-advances-2nd-batch-level-3-approvals/)).

---

## 3. UN R155: Cybersecurity Management System (CSMS)

R155 requires two things:

1. A **CSMS Certificate of Compliance** for the manufacturer. Secondary guidance (e.g. national authority guidelines) states that it is valid for a maximum of three years, with surveillance audits.
2. A **vehicle-type approval** for cybersecurity.

Paragraph 7.2.2.2 lists the processes that the CSMS must contain, items (a)–(h) below. Paragraph 7.2.2.5 adds the management of dependencies on suppliers and service providers. The mapping to ISO/SAE 21434:2021 uses that standard's actual clause titles. ISO/SAE 21434 is the usual way to show compliance, but R155 does not mandate it.

In [ ]:
@dataclass
class CSMSProcess:
    """CSMS process required by UN R155 paragraph 7.2.2.2 (or 7.2.2.5)."""
    ref: str
    process: str
    evidence: List[str]
    iso_21434: str


csms = [
    CSMSProcess("7.2.2.2(a)", "Processes used within the organisation to manage cybersecurity",
                ["Cybersecurity policy", "Roles and responsibilities", "Competence management"],
                "Clause 5 Organizational cybersecurity management; Clause 6 Project dependent cybersecurity management"),
    CSMSProcess("7.2.2.2(b)", "Identification of risks to vehicle types (incl. Annex 5 threats)",
                ["Item definition", "Threat scenarios"], "Clause 9 Concept (item definition); Clause 15 TARA methods"),
    CSMSProcess("7.2.2.2(c)", "Assessment, categorisation and treatment of identified risks",
                ["TARA report", "Risk treatment decisions"], "Clause 15 TARA methods; Clause 9 Concept (cybersecurity goals)"),
    CSMSProcess("7.2.2.2(d)", "Verification that identified risks are appropriately managed",
                ["Verification reports", "Cybersecurity case"], "Clause 10 Product development; Clause 6 (cybersecurity case, assessment)"),
    CSMSProcess("7.2.2.2(e)", "Testing the cybersecurity of a vehicle type",
                ["Penetration test reports", "Fuzzing results"], "Clause 10 Product development (verification); Clause 11 Cybersecurity validation"),
    CSMSProcess("7.2.2.2(f)", "Keeping the risk assessment current",
                ["TARA update records"], "Clause 8 Continual cybersecurity activities; Clause 15"),
    CSMSProcess("7.2.2.2(g)", "Monitoring, detecting and responding to cyber-attacks, threats and vulnerabilities",
                ["Vehicle SOC / PSIRT process", "Incident response plan"], "Clause 8 Continual cybersecurity activities; Clause 13 Operations and maintenance"),
    CSMSProcess("7.2.2.2(h)", "Providing data to support analysis of attempted or successful attacks",
                ["Logging concept", "Forensic data access"], "Clause 8 (cybersecurity monitoring, event evaluation); Clause 13 (incident response)"),
    CSMSProcess("7.2.2.5", "Managing dependencies on contracted suppliers, service providers and sub-organisations",
                ["Cybersecurity interface agreements", "Supplier assessments"], "Clause 7 Distributed cybersecurity activities"),
]

df_csms = pd.DataFrame([{"R155 ref.": c.ref, "Process": c.process, "ISO/SAE 21434:2021": c.iso_21434,
                         "Typical evidence": ", ".join(c.evidence)} for c in csms])
display(df_csms)
clauses = sorted({int(tok) for c in csms for tok in c.iso_21434.replace(";", " ").split() if tok.isdigit()})
print(f"{len(csms)} CSMS processes mapped to ISO/SAE 21434 clauses {clauses}.")

### R155 Annex 5: threats and mitigations

Annex 5 Part A lists threats in **seven** high-level categories, paragraphs 4.3.1–4.3.7. Each category has sub-threats and example attack methods. Physical attacks (for example on the diagnostic port) appear as **examples** inside these categories, not as a separate category. Parts B and C list mitigations. The examples below are a selection, paraphrased.

In [ ]:
annex5 = {
    "4.3.1 Back-end servers related to vehicles in the field": [
        "Abuse of privileges by staff (insider attack)", "Unauthorised internet access to the server",
        "Unauthorised physical access to the server"],
    "4.3.2 Vehicle communication channels": [
        "Spoofing of messages (e.g. V2X, GNSS) by impersonation", "Sybil attack", "Interception / eavesdropping",
        "Denial of service on communication channels"],
    "4.3.3 Update procedures": [
        "Misuse or compromise of update procedures", "Denial of legitimate updates"],
    "4.3.4 Unintended human actions facilitating an attack": [
        "Innocent victim tricked into loading malware or enabling an attack"],
    "4.3.5 External connectivity and connections": [
        "Manipulation of remote-access functions", "Attack via third-party software or apps",
        "Attack via devices connected to external interfaces (USB, OBD port)"],
    "4.3.6 Targets of an attack: data / code": [
        "Extraction of software or data (incl. keys)", "Manipulation of vehicle software or data", "Erasure of data"],
    "4.3.7 Potential vulnerabilities that could be exploited if not sufficiently protected": [
        "Cryptographic weaknesses", "Software or hardware development flaws", "Unintended network transfer between domains"],
}

for cat, threats in annex5.items():
    print(cat)
    for t in threats:
        print(f"   - {t}")
n_examples = sum(len(v) for v in annex5.values())
print(f"\n{len(annex5)} Annex 5 Part A categories; {n_examples} example threats shown (a selection, not the full list).")

---

## 4. UN R156: Software Update Management System (SUMS)

R156 requires a **SUMS** (a process audit of the manufacturer) and a **vehicle-type approval** for software updates. Key ideas:

- **RXSWIN** (R_X Software Identification Number): identifies the type-approval-relevant software of a system governed by UN Regulation X.
- **Impact assessment**: every update is assessed for its effect on type approval, safety and legal requirements.
- **OTA updates** carry extra vehicle-type requirements: safe vehicle state, informing the user, preconditions (e.g. energy), and recovery after failure.

**ISO 24089:2023** (*Road vehicles — Software update engineering*) is the engineering standard most commonly used to implement a SUMS. ISO/SAE 21434 covers the security of the update channel.

In [ ]:
sums = pd.DataFrame([
    ("SUMS (process)", "Identify, version and record all type-approval-relevant software (RXSWIN)", "ISO 24089 (configuration, identification)"),
    ("SUMS (process)", "Assess the impact of each update on type approval, safety and legal compliance", "ISO 24089; ISO 26262-8 Clause 8 change management"),
    ("SUMS (process)", "Check compatibility of the update with the target vehicle configuration", "ISO 24089"),
    ("SUMS (process)", "Protect the authenticity and integrity of updates", "ISO/SAE 21434 (cybersecurity of update channel)"),
    ("Vehicle type", "Protect the update delivery mechanism; record the update in the vehicle", "ISO/SAE 21434; ISO 24089"),
    ("OTA (vehicle type)", "Ensure a safe vehicle state before, during and after the update", "ISO 24089; ISO 26262"),
    ("OTA (vehicle type)", "Inform the user before and after the update", "ISO 24089"),
    ("OTA (vehicle type)", "Check preconditions (e.g. sufficient energy) before starting", "ISO 24089"),
    ("OTA (vehicle type)", "Recover or restore a safe state after a failed update", "ISO 24089; ISO 26262"),
], columns=["R156 area", "Requirement (paraphrased)", "Supporting standard"])
display(sums)
print(sums["R156 area"].value_counts().to_string())

---

## 5. UN R157: Automated Lane Keeping Systems (ALKS, SAE Level 3)

R157 is the first binding international regulation for an SAE Level 3 function. Under the **01 series** (adopted June 2022, applying from January 2023), an ALKS may operate on roads where pedestrians and cyclists are prohibited and opposing traffic is physically separated, at up to **130 km/h**, and may perform **lane changes**.

The regulation requires:
- a **safety management system** audit and an assessment of the safety concept (Annex 4);
- tests (Annex 5);
- driver-availability recognition;
- a **data storage system for automated driving (DSSAD)**;
- compliance with R155 and R156.

**Transition demand and minimum risk manoeuvre (MRM)**, per the consolidated R157 text:
- During a transition demand, the demand is **escalated at the latest 4 s** after it starts (e.g. by adding haptic warnings).
- If the driver has not taken over, an MRM **starts at the earliest 10 s** after the start of the transition demand. It brings the vehicle to a standstill with hazard lights on.
- After a **severe ALKS or vehicle failure**, an MRM may start immediately.

Source: [UN R157 consolidated text (UNECE, 2025)](https://unece.org/sites/default/files/2025-06/R157r1e.pdf).

In [ ]:
ESCALATION_LATEST_S = 4.0   # UN R157: escalate transition demand at the latest 4 s after it starts
MRM_EARLIEST_S = 10.0       # UN R157: MRM starts at the earliest 10 s after start of transition demand


def alks_transition(driver_takeover_s=None, severe_failure=False):
    """Return the sequence of R157 phases (start time in s) after a transition demand is issued at t = 0."""
    if severe_failure:
        return [(0.0, "Severe failure: MRM may start immediately")]
    seq = [(0.0, "Transition demand (optical + acoustic)")]
    if driver_takeover_s is not None and driver_takeover_s < ESCALATION_LATEST_S:
        return seq + [(driver_takeover_s, "Driver takes over; ALKS deactivates")]
    seq.append((ESCALATION_LATEST_S, "Escalation (e.g. haptic) at the latest"))
    if driver_takeover_s is not None and driver_takeover_s < MRM_EARLIEST_S:
        return seq + [(driver_takeover_s, "Driver takes over; ALKS deactivates")]
    seq.append((MRM_EARLIEST_S, "No takeover: MRM starts (earliest)"))
    return seq


cases = {"Driver responds at 2.7 s": alks_transition(2.7),
         "Driver responds at 7 s": alks_transition(7.0),
         "No driver response": alks_transition(None),
         "Severe ALKS failure": alks_transition(severe_failure=True)}

fig, ax = plt.subplots(figsize=(12, 4.5))
# Take-over time literature: mean 2.72 s, SD 1.45 s across 129 studies (Zhang et al., 2019)
ax.axvspan(2.72 - 1.45, 2.72 + 1.45, color=OI["sky"], alpha=0.3, label="Mean ± 1 SD take-over time (Zhang et al. 2019)")
for i, (name, seq) in enumerate(cases.items()):
    y = len(cases) - 1 - i
    times = [t for t, _ in seq]
    ax.plot([0, max(max(times), 0.3)], [y, y], color=OI["grey"], lw=2)
    for t, label in seq:
        colour = OI["vermillion"] if "MRM" in label else OI["green"] if "takes over" in label else OI["blue"]
        ax.scatter(t, y, color=colour, s=60, zorder=3, edgecolor="black")
        ax.annotate(label, (t, y), xytext=(3, 7), textcoords="offset points", fontsize=7)
ax.axvline(ESCALATION_LATEST_S, ls=":", color="black")
ax.axvline(MRM_EARLIEST_S, ls="--", color="black")
ax.set_yticks(range(len(cases)))
ax.set_yticklabels(list(cases)[::-1])
ax.set_xlim(-0.5, 14)
ax.set_xlabel("Time since start of transition demand (s)")
ax.set_title("UN R157 transition demand: escalation ≤ 4 s, MRM no earlier than 10 s")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

for name, seq in cases.items():
    print(f"{name}: " + " -> ".join(f"{label} @ {t:.1f} s" for t, label in seq))

**Human factors.** In a meta-analysis of 129 studies, the mean take-over time was 2.72 s (SD 1.45 s), and it depended strongly on the time budget and urgency (Zhang et al., 2019). Non-critical take-overs can take much longer. The R157 ≥ 10 s window before an MRM is therefore generous for an attentive driver, but it is not a guarantee of safe take-over quality.

---

## 6. EU AI Act: how it reaches vehicles

**Correct classification.** AI systems that are safety components of vehicles are **high-risk via Art. 6(1) and Annex I, Section B**. Annex I Section B lists the vehicle type-approval acts, including Reg. (EU) 2018/858 and Reg. (EU) 2019/2144.

**Indirect application.** Under **Art. 2(2)**, only a few provisions of the AI Act apply directly to these products: Art. 6(1), Arts 102–109 and Art. 112. Arts 102–109 amend the sectoral acts (for motor vehicles, Art. 107 amends Reg. (EU) 2018/858 and Art. 109 amends Reg. (EU) 2019/2144) so that the Commission must take the Chapter III, Section 2 requirements into account when it adopts future **delegated or implementing acts** for vehicles. Vehicle AI therefore gets AI-Act requirements **through type approval**, not through stand-alone AI Act conformity assessment.

**Common error.** Annex III point 2 covers AI used as safety components in the management and operation of critical digital infrastructure, road traffic, and the supply of water, gas, heating and electricity. That is **infrastructure** (e.g. traffic management), not vehicles.

**Dates (as of October 2026).** The Digital Omnibus on AI, **Regulation (EU) 2026/1744** (in force 27 July 2026), moved the high-risk application dates:
- Annex III systems: **2 December 2027**.
- Art. 6(1) / Annex I product-related systems: **2 August 2028**.

For vehicles (Section B), the practical trigger remains the adoption of the vehicle delegated or implementing acts. Sources: [EUR-Lex 2026/1744](https://eur-lex.europa.eu/eli/reg/2026/1744/oj/eng); [AI Act Art. 2](https://artificialintelligenceact.eu/article/2/).

In [ ]:
ai_act = {
    "Prohibited (Art. 5)": {
        "examples": ["Social scoring", "Emotion recognition in workplaces and education"],
        "automotive": "Not expected for vehicle functions; check driver-facing features against Art. 5"},
    "High-risk via Art. 6(1) + Annex I": {
        "examples": ["Safety components of products under Union harmonisation legislation"],
        "automotive": "Vehicle AI safety components (Annex I Section B: Reg. 2018/858, 2019/2144); "
                      "requirements arrive through vehicle delegated/implementing acts (Art. 2(2); Arts 107 and 109 amend 2018/858 and 2019/2144)"},
    "High-risk via Annex III": {
        "examples": ["Critical infrastructure incl. road-traffic management (point 2)", "Emotion recognition (point 1(c))"],
        "automotive": "Road-side traffic-management AI (infrastructure), not in-vehicle ADS"},
    "Transparency obligations (Art. 50)": {
        "examples": ["Chatbots", "Deep fakes", "Deployers of emotion-recognition systems"],
        "automotive": "In-car voice assistants that interact with people"},
    "Minimal risk": {
        "examples": ["Spam filters"],
        "automotive": "Many infotainment features"},
}
for level, d in ai_act.items():
    print(level)
    print(f"   examples   : {', '.join(d['examples'])}")
    print(f"   automotive : {d['automotive']}")
print("\nNote: driver drowsiness and distraction warning (DDAW/ADDW) is regulated under Reg. (EU) 2019/2144; "
      "it is not emotion recognition.")

### Chapter III, Section 2 requirements: awareness mapping

> These articles are **not directly applicable** to vehicles (Art. 2(2)). The table shows which existing automotive practice would most likely provide evidence once vehicle acts take them into account. It is the author's interpretation, not an official mapping.

In [ ]:
art_map = pd.DataFrame([
    ("Art. 9", "Risk management system", "ISO 26262 HARA, ISO 21448, ISO/PAS 8800 (AI safety lifecycle)"),
    ("Art. 10", "Data and data governance (data sets 'to the best extent possible, free of errors and complete')",
     "ISO/PAS 8800 data-set requirements"),
    ("Art. 11", "Technical documentation", "Type-approval information document; safety and cybersecurity cases"),
    ("Art. 12", "Record-keeping (logging)", "EDR (Reg. 2019/2144); DSSAD (UN R157, EU 2022/1426); GB 44497 in China"),
    ("Art. 13", "Transparency and instructions for use", "Owner's manual; HMI and ODD information to the driver/user"),
    ("Art. 14", "Human oversight", "Driver role per SAE J3016 level; R157 driver-availability and transition demand"),
    ("Art. 15", "Accuracy, robustness and cybersecurity", "ISO 21448 / ISO/PAS 8800 robustness; ISO/SAE 21434 and UN R155"),
], columns=["Article", "Requirement", "Likely automotive evidence"])
display(art_map)

---

## 7. Standards → regulation mapping

Regulations are mostly **technology-neutral** and do not mandate standards. The matrix shows how often a standard serves as the main body of evidence ("Primary") or as supporting evidence ("Supporting") for each regulation, in the author's judgement.

ISO/PAS 8800 is **not** a harmonised standard under the AI Act. CEN-CENELEC JTC 21 is developing the harmonised standards.

In [ ]:
standards = ["ISO 26262", "ISO 21448", "ISO/SAE 21434", "ISO 24089", "ISO/PAS 8800", "ISO/TS 5083", "UL 4600"]
regulations = ["UN R155", "UN R156", "UN R157", "UN ADS Reg.", "EU 2022/1426", "EU AI Act"]
# 0 = no direct relation, 1 = supporting evidence, 2 = primary evidence (author's interpretation)
mapping = np.array([
    [1, 1, 2, 2, 2, 0],   # ISO 26262
    [0, 0, 2, 2, 2, 1],   # ISO 21448
    [2, 1, 1, 1, 1, 1],   # ISO/SAE 21434
    [0, 2, 1, 1, 1, 0],   # ISO 24089
    [0, 0, 1, 1, 1, 1],   # ISO/PAS 8800 (only where ML is used; not harmonised under AI Act)
    [0, 0, 1, 2, 1, 0],   # ISO/TS 5083
    [0, 0, 0, 1, 1, 0],   # UL 4600
])
labels = {0: "-", 1: "Supporting", 2: "Primary"}

fig, ax = plt.subplots(figsize=(11, 5.5))
im = ax.imshow(mapping, cmap="viridis", vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(len(regulations)))
ax.set_xticklabels(regulations, rotation=30, ha="right")
ax.set_yticks(range(len(standards)))
ax.set_yticklabels(standards)
for i in range(len(standards)):
    for j in range(len(regulations)):
        ax.text(j, i, labels[mapping[i, j]], ha="center", va="center", fontsize=8,
                color="black" if mapping[i, j] == 2 else "white")
ax.set_title("Standards as evidence for regulations (author's interpretation)")
plt.tight_layout()
plt.show()

for j, reg in enumerate(regulations):
    prim = [standards[i] for i in range(len(standards)) if mapping[i, j] == 2]
    print(f"{reg:<13} primary evidence: {', '.join(prim) if prim else 'none'}")

---

## 8. Type-approval process with CSMS / SUMS

> **Illustrative data:** the durations below are indicative planning values chosen for teaching; they do not come from any regulation and vary widely between manufacturers and authorities.

In [ ]:
phases = pd.DataFrame([
    ("CSMS preparation", 0, 4), ("CSMS audit -> Certificate of Compliance", 4, 2),
    ("SUMS preparation", 3, 3), ("SUMS audit -> Certificate of Compliance", 6, 1.5),
    ("Vehicle-type TARA and treatment", 2, 6), ("Security testing (technical service)", 5, 4),
    ("Vehicle-type assessment (R155/R156)", 8, 3), ("Approval granted", 11, 1),
], columns=["phase", "start_month", "duration_months"])

fig, ax = plt.subplots(figsize=(12, 4.5))
cols = [OI["blue"], OI["blue"], OI["green"], OI["green"], OI["orange"], OI["orange"], OI["purple"], OI["vermillion"]]
for i, r in phases.iterrows():
    ax.barh(i, r["duration_months"], left=r["start_month"], color=cols[i], alpha=0.85, edgecolor="black")
ax.set_yticks(range(len(phases)))
ax.set_yticklabels(phases["phase"])
ax.invert_yaxis()
ax.set_xlabel("Months from start (indicative)")
ax.set_title("Indicative R155/R156 approval plan (illustrative durations)")
plt.tight_layout()
plt.show()

end = (phases["start_month"] + phases["duration_months"]).max()
print(f"Indicative plan length: {end:.0f} months (illustrative). "
      "The CSMS Certificate of Compliance is limited to max. 3 years and is subject to surveillance; "
      "after approval the manufacturer reports monitoring outcomes to the authority at least annually (R155).")

---

## 9. Worked checklist: an SAE Level 3 motorway system in the EU

**Scenario.** An OEM wants EU whole-vehicle type approval for an M1 car with an ALKS-type motorway chauffeur. It works up to 130 km/h, performs lane changes, and uses an ML-based camera perception stack. What does the OEM need to show? The checklist is a teaching aid, not an exhaustive legal list.

In [ ]:
@dataclass
class ChecklistItem:
    area: str
    item: str
    basis: str
    evidence: str
    standards: str = "-"


l3_checklist = [
    ChecklistItem("Approval route", "EU whole-vehicle type approval incl. all applicable GSR items",
                  "Reg. (EU) 2018/858; Reg. (EU) 2019/2144", "Information document; approvals of all systems"),
    ChecklistItem("ALKS", "UN R157 approval (01 series): ODD declaration, max. 130 km/h, lane-change strategy",
                  "UN R157", "ODD and system description; technical-service test reports (Annex 5)", "ISO 34503 (ODD description)"),
    ChecklistItem("ALKS", "Safety management system and safety concept audit",
                  "UN R157 Annex 4", "Safety case; HARA; FMEA/FTA; SOTIF analysis; validation report",
                  "ISO 26262, ISO 21448, ISO/TS 5083"),
    ChecklistItem("ALKS", "Transition demand <= 4 s escalation, MRM >= 10 s, driver-availability recognition",
                  "UN R157", "HMI specification; driver-monitoring validation; test results", "ISO 21448 (foreseeable misuse)"),
    ChecklistItem("ALKS", "DSSAD records (activation, transition demands, MRM, takeover)",
                  "UN R157", "DSSAD specification and read-out procedure"),
    ChecklistItem("Perception (ML)", "Evidence that ML perception meets its safety requirements within the ODD",
                  "UN R157 Annex 4 (safety concept)", "Data-set specification, ODD coverage, robustness and uncertainty evaluation",
                  "ISO/PAS 8800, ISO 21448"),
    ChecklistItem("Scenarios", "Scenario-based validation incl. critical scenarios and simulation credibility",
                  "UN R157 Annex 4/5", "Scenario catalogue; simulation validation report", "ISO 34502"),
    ChecklistItem("Cybersecurity", "CSMS Certificate of Compliance + vehicle-type approval",
                  "UN R155", "CSMS certificate; TARA; cybersecurity case; test reports", "ISO/SAE 21434"),
    ChecklistItem("Software update", "SUMS Certificate + vehicle-type approval; RXSWIN for ALKS software",
                  "UN R156", "SUMS certificate; RXSWIN register; update impact procedure", "ISO 24089"),
    ChecklistItem("Data recording", "Event data recorder (EDR)", "Reg. (EU) 2019/2144", "EDR approval"),
    ChecklistItem("Liability", "Product liability exposure for software and updates (products placed on the market after 9 Dec 2026)",
                  "Directive (EU) 2024/2853 (national transposition)", "Defect-prevention evidence; update and monitoring records"),
    ChecklistItem("National law", "Rules on use of automated driving functions by the driver (e.g. Germany StVG §1a/1b)",
                  "National road-traffic law", "Owner's manual; driver information"),
    ChecklistItem("AI Act", "Track vehicle delegated/implementing acts that integrate AI Act requirements",
                  "Reg. (EU) 2024/1689 Art. 2(2), Arts 107 and 109; Reg. (EU) 2026/1744", "Gap analysis vs Chapter III Section 2",
                  "ISO/PAS 8800"),
    ChecklistItem("In service", "Field monitoring, vulnerability management, annual R155 monitoring report, update impact checks",
                  "UN R155 / R156; ISO 21448 Clause 13", "Monitoring reports; incident records; SPIs", "ISO/SAE 21434, ISO 21448, UL 4600 (SPIs)"),
]

df_l3 = pd.DataFrame([vars(i) for i in l3_checklist])
display(df_l3)

per_area = df_l3["area"].value_counts()
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(per_area.index[::-1], per_area.values[::-1], color=OI["blue"])
ax.set_xlabel("Number of checklist items")
ax.set_title("L3 motorway system (EU): checklist items per area")
plt.tight_layout()
plt.show()

print(f"{len(df_l3)} checklist items across {df_l3['area'].nunique()} areas.")
std_mentions = pd.Series([s.split("(")[0].strip() for x in df_l3["standards"] if x != "-" for s in x.split(",")]).value_counts()
print("Standards cited as evidence:", ", ".join(f"{k} ({v})" for k, v in std_mentions.items()))

---

## 10. Regional comparison (as of October 2026)

| Region | Approval model | Cybersecurity / software update | Automated driving |
|---|---|---|---|
| **UNECE (1958 Agreement)** | Type approval, mutual recognition | UN R155 / R156 | UN R157 (L3 ALKS, ≤ 130 km/h); UN ADS Regulation adopted June 2026; UN R171 (DCAS, Level 2) |
| **EU** | Type approval (2018/858) | R155/R156 mandatory for all new vehicles since July 2024 | R157; Reg. 2022/1426 for small-series fully automated vehicles; AI Act indirectly (dates moved by 2026/1744) |
| **Germany** | EU type approval + national operation rules | as EU | Level 4 operation under StVG §§1d–1l + AFGBV with technical supervision |
| **UK** | GB type approval | R155/R156-equivalent requirements for new types from 1 June 2026 | Automated Vehicles Act 2024; full implementation targeted for H2 2027; APS pilots 2026 |
| **US** | Self-certification (FMVSS) | NHTSA cybersecurity best practices (2022, guidance) | SGO crash reporting; Part 555 / exemption programme; 2026 FMVSS proposals for vehicles without manual controls |
| **China** | Product access (MIIT) + CCC + mandatory GB | GB 44495-2024, GB 44496-2024 (from 1 Jan 2026) | GB 44497-2024 (DSSAD); first conditional L3 approvals Dec 2025 |

---

## Key takeaways

- **Type approval** (UNECE, EU, China) checks compliance before sale, through authorities, technical services and management-system audits. The **US** relies on **self-certification** plus enforcement and reporting.
- **R155/R156** have applied to all new vehicles in the EU since July 2024. ISO/SAE 21434 and ISO 24089 are the usual engineering routes, with correct clause mapping (TARA = Clause 15; monitoring = Clause 8; incident response = Clause 13).
- **R157** allows L3 motorway driving up to 130 km/h with lane change (01 series). The transition demand escalates **≤ 4 s**; the MRM starts **≥ 10 s** unless a severe failure occurs.
- **Level 4** has legal routes: EU 2022/1426 (small series), the German AFGBV, and since June 2026 the **UN ADS Regulation**, built on a safety management system, safety case and in-service monitoring.
- The **EU AI Act** reaches vehicles through **Annex I Section B** and future vehicle acts (Art. 2(2)), not through Annex III point 2. Reg. (EU) 2026/1744 moved the high-risk dates to Dec 2027 (Annex III) and Aug 2028 (Annex I).
- Liability is changing too: the new **Product Liability Directive (EU) 2024/2853** treats software as a product from 9 December 2026.

## Self-assessment

1. What is the difference between type approval and self-certification? Name the actors in an EU approval.
2. Which ISO/SAE 21434 clauses support R155 7.2.2.2(c) and (g)?
3. In R157, when must the transition demand be escalated, and when may an MRM start?
4. Why is "ADAS are high-risk under AI Act Annex III point 2" wrong?
5. Name two legal routes for deploying a Level 4 vehicle in Germany in 2026.
6. Which standard is the primary engineering reference for a SUMS?

<details>
<summary>Answers</summary>

1. In type approval, an approval authority grants approval before sale, based on tests and audits by a technical service. In self-certification (US), the manufacturer certifies FMVSS compliance itself and NHTSA enforces after sale. EU actors: the manufacturer, the type-approval authority (e.g. KBA) and the technical service (e.g. TÜV/DEKRA), with market surveillance after approval.
2. (c) Clause 15 (TARA methods) and Clause 9 (concept: cybersecurity goals). (g) Clause 8 (continual activities: monitoring, event evaluation, vulnerability management) and Clause 13 (incident response, updates).
3. Escalation at the latest 4 s after the start of the transition demand. The MRM starts at the earliest 10 s after the start if the driver has not taken over, or immediately after a severe ALKS or vehicle failure.
4. Annex III point 2 covers critical infrastructure, including road-traffic management. Vehicle AI is high-risk via Art. 6(1) and Annex I Section B, and Art. 2(2) limits direct application to a few articles; the requirements arrive through vehicle acts.
5. Any two of: national approval under StVG §§1d–1l + AFGBV (operating area, technical supervisor); EU type approval under Reg. (EU) 2022/1426 (small series); the UN ADS Regulation route once applied in the EU.
6. ISO 24089:2023 (software update engineering), with ISO/SAE 21434 for the security of the update channel.
</details>

## References

- UNECE. UN Regulation No. 155 (Cyber security and CSMS); No. 156 (Software update and SUMS); No. 157 (ALKS), consolidated text: https://unece.org/sites/default/files/2025-06/R157r1e.pdf
- UNECE (24 June 2026). *UNECE adopts first-ever global rules allowing fully autonomous vehicles*. https://unece.org/sustainable-development/press/unece-adopts-first-ever-global-rules-allowing-fully-autonomous
- Regulation (EU) 2018/858; Regulation (EU) 2019/2144 (General Safety Regulation). https://eur-lex.europa.eu/eli/reg/2019/2144/oj
- Commission Implementing Regulation (EU) 2022/1426 (ADS of fully automated vehicles). https://eur-lex.europa.eu/eli/reg_impl/2022/1426/oj ; amended by (EU) 2026/481.
- Regulation (EU) 2024/1689 (AI Act). https://eur-lex.europa.eu/eli/reg/2024/1689/oj ; Regulation (EU) 2026/1744 (Digital Omnibus on AI). https://eur-lex.europa.eu/eli/reg/2026/1744/oj/eng
- Directive (EU) 2024/2853 on liability for defective products. https://eur-lex.europa.eu/eli/dir/2024/2853/oj
- Germany: Straßenverkehrsgesetz §§1d–1l; Autonome-Fahrzeuge-Genehmigungs-und-Betriebs-Verordnung (AFGBV), 24 June 2022.
- UK: Automated Vehicles Act 2024. https://www.legislation.gov.uk/ukpga/2024/10/contents
- NHTSA: Standing General Order 2021-01 (Third Amendment, effective 16 June 2025). https://www.nhtsa.gov/laws-regulations/standing-general-order-crash-reporting
- China: GB 44495-2024, GB 44496-2024, GB 44497-2024 (mandatory from 1 January 2026).
- ISO/SAE 21434:2021; ISO 24089:2023 *Road vehicles — Software update engineering*.
- Zhang, B., de Winter, J., Varotto, S., Happee, R., Martens, M. (2019). Determinants of take-over time from automated driving: A meta-analysis of 129 studies. *Transportation Research Part F*, 64, 285–307. https://doi.org/10.1016/j.trf.2019.04.020

---
**Navigation:** ← [Standards integration](01_standards_integration.ipynb) · [Module index](README.md) · [Standards gaps](03_standards_gaps.ipynb) →